# ОПИСАНИЕ НОУТБУКА

Предполагается тестирование модели SmolVLM 
загрузка модели с HuggingFaceTB
Специфика модели предусматривает промпт, который передается в структуру message, после чего формируется описание изображения.

## ПАЙПЛАЙН включает в себя
- модель для получения текстовых эмбедингов для размеченных картинок
- получения описания по картинке
- получения эмбеденга описания
- расчет расстояния (L2-эвклидова) между описанием и всеми размечеными картинками
- выбор картинки с наименьшим расстоянием
- формирование списка предсказаний
- расчет метрик классификации на основе списка предсказаний и сравнения его с разметокой

In [1]:
import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image
import torch
from pathlib import Path
import sklearn
from tqdm import tqdm
import PIL
import pandas as pd
import numpy as np
from typing import List

#from smolvlm.translate import translate_en_to_ru # использовал для перевода в локальном тесте
from smolvlm.imagetotextHF import image_to_text_pipeline

c:\Users\User\anaconda3\envs\RESEARCHPY312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используем устройство: cpu


In [2]:
# можно вынести в config
MODEL_PATH = "HuggingFaceTB/SmolVLM-500M-Instruct"
PROMPT = "Describe the product in the image for an online sales listing. First, write a 2–3 sentence description. Then add a line starting with 'Keywords:' followed by 8–15 comma-separated keywords about type, color, material, style, use cases, and target audience."
# это актуально для модели smolvlm
MESSAGES = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": f"{PROMPT}"},
        ],
    }]

In [3]:
# # works only with old version of transformers
# !pip install -U transformers==4.37.1

In [4]:
# # there is a bug in ipykernel that prevents downloading of BAAI/bge-large-en-v1.5, so we need to be sure that it is updated
# # https://github.com/huggingface/xet-core/issues/526
# !pip install -U ipykernel>=7.1.0

In [5]:
def get_metrics_txt_emb(text):

    """
    Generates a normalized text embedding using BGE-large-en-v1.5 model.
    
    Args:
        text (str): Input text to generate embedding for
        
    Returns:
        numpy.ndarray: Normalized embedding vector (shape: [1024])
    """
    METRICS_TXT_EMB_MODEL_NAME=r"smolvlm/bge"

    tokenizer = transformers.AutoTokenizer.from_pretrained(METRICS_TXT_EMB_MODEL_NAME)
    model = transformers.AutoModel.from_pretrained(
        METRICS_TXT_EMB_MODEL_NAME,
        dtype=torch.float32
    )
    model.eval()
    
    encoded_input = tokenizer([text], padding=True, truncation=True, return_tensors='pt')
    
    with torch.no_grad():
        model_output = model(**encoded_input)
        sentence_embeddings = model_output[0][:, 0] 
    

    normalized = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)[0]
    return normalized.detach().cpu().numpy()

## Load Data

In [6]:
def load_image_markup(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        while True:
            line = f.readline()
            if len(line) == 0:
                break
            sep_ind = line.find("\"")
            path = ("../" + line[:sep_ind]).strip()
            desc = line[sep_ind + 1:].strip()[:-1]
            data.append((path, desc))
    return data

In [7]:
# функция для получения эмбедингов для текста (описание картинки)

def text_vector_for_image_description(markup):
    """ 
    получаем вектор (эмбединг) для текстового описания картинки
    и еще убираем дубликаты описаний картинок
    markup : List(tuple(str)) это список кортежей с данными о изображении и описании изображения
    по типу [('../data/pictures/1.jpg',
            "women's beige demi-season coat, women's denim jacket"),
            .....
            ]
    """

    text_list = [markup_line[1].lower() for markup_line in markup]

    # some descriptions are the same or mostly same (start from the same words)
    unique_indices = []
    for idx in range(len(text_list)):
        if idx == 0:
            unique_indices.append(idx)
            continue
        prev_started = any(txt.startswith(text_list[idx]) for txt in text_list[:idx])
        current_started = any(text_list[idx].startswith(txt) for txt in text_list[:idx])
        if prev_started or current_started:
            continue
        unique_indices.append(idx)
    unique_text_list = [text_list[idx] for idx in unique_indices]
    emb_unique_text_list = [get_metrics_txt_emb(txt) for txt in unique_text_list]

    return unique_indices, unique_text_list, emb_unique_text_list

In [8]:
def predict_vs_true_text(markup,
                            unique_indices,
                         unique_text_list,
                         emb_unique_text_list,
                         model_path:str = MODEL_PATH,
                         message: str = MESSAGES):
    
    text_list = [markup_line[1].lower() for markup_line in markup]
    y_true_list = [] 
    distance_list = []
    for img_idx, markup_line in tqdm(enumerate(markup), total=len(text_list)):        
        # это кастомная функция для генерации текста от модели
        decoded_out = image_to_text_pipeline(model_path = model_path , image_path = markup_line[0], message=message)
        # получение эмбединга к описанию
        emb_decoded_out = get_metrics_txt_emb(decoded_out[0].split('Assistant: ')[1]) #decoded_out[0].split('Assistant: ')[1] - это специфика ответа модели
        # получение значения эвклидового расстояния между эмбедингами дял тектса полученного описания до каждого описания в датасете
        distance_per_image = [np.linalg.norm(emb_unique_text - emb_decoded_out) for emb_unique_text in emb_unique_text_list]

        if img_idx not in unique_indices:
            assert markup_line[1].lower() == text_list[img_idx]
            txt_to_find = markup_line[1].lower()
            u_img_idx = -1
            for i in range(len(unique_text_list)):
                if unique_text_list[i].startswith(txt_to_find) or txt_to_find.startswith(unique_text_list[i]):
                    u_img_idx = i
                    break
            assert u_img_idx >= 0
            assert u_img_idx < img_idx
        else:
            u_img_idx = unique_indices.index(img_idx)

        y_true_list += [1 if idx == u_img_idx else 0 for idx in range(len(unique_text_list))]
        distance_list += distance_per_image

    return y_true_list, distance_list

In [9]:
def get_classification_metrics(markup, y_true_vec, logit_vec, unique_text_list, distance_list, max_distance):

    text_list = [markup_line[1].lower() for markup_line in markup]


    assert y_true_vec.shape == logit_vec.shape
    assert logit_vec.min() >= 0
    assert logit_vec.max() <= 1
    assert np.allclose(np.sort(np.unique(y_true_vec)), np.array([0, 1]))

    # https://stats.stackexchange.com/q/287117/
    prevalence = np.count_nonzero(y_true_vec == 1, keepdims=False)/len(y_true_vec)
    assert prevalence > 0
    fpr_vec, tpr_vec, thr_vec = sklearn.metrics.roc_curve(y_true_vec, logit_vec)
    recall_vec = tpr_vec
    tnr_vec = 1 - fpr_vec

    zero_div_idxs = np.where((recall_vec*prevalence) + ((1 - tnr_vec)*(1 - prevalence)) == 0)[0]
    if zero_div_idxs.size > 0:
        # to avoid zero-division warning from numpy below
        recall_vec[zero_div_idxs] += 1e-8
    precision_vec = (recall_vec*prevalence)/((recall_vec*prevalence) + ((1 - tnr_vec)*(1 - prevalence)))

    zero_div_idxs = np.where(precision_vec + recall_vec == 0)[0]
    if zero_div_idxs.size > 0:
        # to avoid zero-division warning from numpy below
        precision_vec[zero_div_idxs] += 1e-8
        recall_vec[zero_div_idxs] += 1e-8
    f1_vec = 2*(precision_vec*recall_vec)/(precision_vec + recall_vec)

    opt_thr = thr_vec[np.argmax(f1_vec)]
    if np.isinf(opt_thr):
        opt_thr = 1

    # using ">=" below instead of ">" is extremely important, because sklearn cn return edge values for threshold
    tp = np.count_nonzero((logit_vec >= opt_thr) & (y_true_vec == 1))
    fp = np.count_nonzero((logit_vec >= opt_thr) & (y_true_vec == 0))
    tn = np.count_nonzero((logit_vec < opt_thr) & (y_true_vec == 0))
    fn = np.count_nonzero((logit_vec < opt_thr) & (y_true_vec == 1))

    tp_list = [
        (distance_list[tp_idx], text_list[tp_idx % len(unique_text_list)])
        for tp_idx in np.nonzero((logit_vec >= opt_thr) & (y_true_vec == 1))[0]
    ]
    tp_list = [(sum(dist for dist, txt in tp_list if txt == u_txt), u_txt) for u_txt in set([txt for dist, txt in tp_list])]
    tp_list.sort(key=lambda item: item[0])
    fp_list = [
        (distance_list[fp_idx], text_list[fp_idx % len(unique_text_list)])
        for fp_idx in np.nonzero((logit_vec >= opt_thr) & (y_true_vec == 0))[0]
    ]
    fp_list = [(sum(dist for dist, txt in fp_list if txt == u_txt), u_txt) for u_txt in set([txt for dist, txt in fp_list])]
    fp_list.sort(key=lambda item: item[0])
    opt_thr = max_distance - (float(opt_thr)*max_distance)

    return tp, fp, tn, fn, opt_thr, tp_list, fp_list

## Metrics

In [10]:
def prediccion(markup, **kwargs):

    """
    Calculates VLM retrieval metrics using distance-based ranking and optimal F1 threshold.
    
    Args:
        markup (list): List of [image_path, text_description] pairs
        model: Vision-Language Model for text generation
        processor_obj: Model processor/tokenizer
        
    Returns:
        tuple: (tp, fp, tn, fn, opt_thr, tp_list, fp_list)
    """

    
    unique_indices, unique_text_list, emb_unique_text_list = text_vector_for_image_description(markup)

    if kwargs:
        model_path = kwargs.get('model_path', MODEL_PATH)
        message = kwargs.get('message', MESSAGES)
    else:
        model_path = MODEL_PATH
        message = MESSAGES


    y_true_list, distance_list = predict_vs_true_text(markup,
                            unique_indices,
                         unique_text_list,
                         emb_unique_text_list,
                         model_path=model_path,
                         message=message)
    
    return y_true_list, distance_list, unique_text_list

def get_values_conf_matrix(markup, y_true_list, distance_list, unique_text_list):
    y_true_vec = np.array(y_true_list)
    max_distance = max(distance_list)
    logit_vec = np.array([(max_distance - dist)/max_distance for dist in distance_list])
    
    return get_classification_metrics(markup, y_true_vec, logit_vec, unique_text_list, distance_list, max_distance)

------

# MAIN CALULATION

In [11]:
img_markup = load_image_markup("../data/matching_images.txt")

In [12]:
y_true_list, distance_list, unique_text_list = prediccion(img_markup)

  0%|          | 0/311 [00:00<?, ?it/s]

Answer: To create a detailed description of the image, I will consider the following keywords:
1. **Product Description**: A pair of beige, double-breasted, wide-necked, long


  0%|          | 1/311 [01:12<6:13:32, 72.30s/it]

Answer: Description: A collection of fabric swatches, including a floral-patterned dress, a light green fabric, a floral-patterned fabric, a black fabric, and a blue fabric. The fabric


  1%|          | 2/311 [02:20<5:59:05, 69.72s/it]

...

 99%|█████████▉| 309/311 [2:15:10<00:51, 25.90s/it]

Answer: Black shoes with white strips and 100s written on the side.


100%|█████████▉| 310/311 [2:15:32<00:24, 24.65s/it]

Answer: Black Adidas shoes with white laces.


100%|██████████| 311/311 [2:15:55<00:00, 26.22s/it]


In [14]:
tp, fp, tn, fn, thr, tp_list, fp_list = get_values_conf_matrix(img_markup, y_true_list, distance_list, unique_text_list)

if 2*tp + fp + fn > 0:
    f1 = 2*tp/(2*tp + fp + fn)
else:
    f1 = 0

if tp + fp + tn + fn > 0:
    acc = (tp + tn)/(tp + fp + tn + fn)
else:
    acc = 0

print("TRUE POSITIVES:\n\t" + "\n\t".join(f"({dist:.1f}) {txt}" for dist, txt in tp_list) + "\n\n")
print("FALSE POSITIVES:\n\t" + "\n\t".join(f"({dist:.1f}) {txt}" for dist, txt in fp_list) + "\n\n")

print(f"Model: SmolVLM-500M-Instruct")
print(f"TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
print(f"Threshold: {thr}")
print(f"Accuracy: {acc}")
print(f"F1: {f1}")

TRUE POSITIVES:
	(0.3) a bag of children's toys, a children's helicopter, a children's book, an agry birds figurine
	(0.5) metal cage for birds
	(0.5) double bed with mattress and wooden headboard
	(0.6) porcelain teapots, white with floral print and black, sugar bowls, tea cups and saucers
	(0.6) black and grey kittens are eating from the plate
	(0.6) 3 doors wooden cabinet, armchair, wooden chairs
	(0.6) warm zipped jacket with a hood
	(0.6) black women's ankle boots with heels, white high-heeled shoes
	(0.6) women's warm down jacket
	(0.6) red and yellow plastic children's chairs
	(0.6) khaki zip-up hooded sweatshirt
	(0.6) black sleeveless dress for girls
	(0.6) brown plastic baseboards
	(0.6) dark blue women's dress
	(0.6) pink rubber boots for girls with unicorns, black patent leather shoes for girls, pink summer sandals for girls, burgundy summer shoes
	(0.6) dark blue dress for girls with a star print
	(0.6) baby diapers
	(0.6) warm children's boots for boys with a car print
	(